# Evaluation — How the Model Actually Performed

All real numbers. No polished metrics here — just what the model does, where it struggles, and an honest explanation of why the R² is what it is.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, pickle

import tensorflow as tf
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')
os.makedirs('../outputs/figures', exist_ok=True)

train_df = pd.read_csv('../data/processed/train.csv')
test_df  = pd.read_csv('../data/processed/test.csv')
with open('../data/processed/scaler_y.pkl','rb') as f: scaler_y = pickle.load(f)
with open('../outputs/models/dummy.pkl','rb') as f: dummy = pickle.load(f)
with open('../outputs/models/linear_regression.pkl','rb') as f: lr = pickle.load(f)
nn = tf.keras.models.load_model('../outputs/models/best_nn.keras')

feat_cols = [c for c in train_df.columns if c not in ['loan_amnt_scaled','loan_amnt_raw']]
X_test    = test_df[feat_cols].values
y_test    = test_df['loan_amnt_raw'].values

y_pred_dm   = scaler_y.inverse_transform(dummy.predict(X_test).reshape(-1,1)).flatten()
y_pred_lr   = lr.predict(X_test)
y_pred_sc   = nn.predict(X_test, verbose=0).flatten()
y_pred_nn   = scaler_y.inverse_transform(y_pred_sc.reshape(-1,1)).flatten()

print(f'Test set: {len(y_test):,} samples  |  Loan range: ${y_test.min():,}–${y_test.max():,}')

## Full Metrics Table

> **Why RMSE and not accuracy?**
> This is a regression problem — we're predicting a continuous dollar amount,
> not a class label. RMSE penalises large errors more than small ones (important
> since a $15k miss on a $20k loan is much worse than a $500 miss). R² tells us
> what fraction of loan amount variance the model explains — 0 means no better
> than the mean, 1 means perfect.

In [ ]:
def metrics(y_true, y_pred, name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return {'Model': name, 'RMSE': f'${rmse:,.0f}', 'MAE': f'${mae:,.0f}',
            'R²': f'{r2:.4f}', 'MAPE': f'{mape:.1f}%'}

rows = [
    metrics(y_test, y_pred_dm, 'Dummy (mean = $9,589)'),
    metrics(y_test, y_pred_lr, 'Linear Regression'),
    metrics(y_test, y_pred_nn, 'Neural Network'),
]
pd.DataFrame(rows)

### What these numbers actually mean

**Dummy**: RMSE of $6,347, R² ≈ 0. Predicts $9,589 every time. This is the absolute floor.

**Linear Regression**: RMSE $5,916, R² 0.131. Modest improvement — income, grade, and intent give it some signal, but not much.

**Neural Network**: RMSE $5,571, R² 0.230. The best of the three, but R² of 0.23 means it explains only 23% of loan amount variance. The other 77% is personal choice — how much the borrower decided they needed — which no feature in this dataset can capture.

**This is not a failure.** A R² of 0.23 on loan amount is actually reasonable given what the features measure. The features describe creditworthiness, not intent. If you had features like: 'what home renovation is planned', 'what medical procedure', 'what debt needs consolidating' — R² would be much higher. But those features don't exist in this dataset.

## Actual vs Predicted — The Honest Scatter

In [ ]:
np.random.seed(42)
sample = np.random.choice(len(y_test), 1500, replace=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, y_pred, name, color in [
    (axes[0], y_pred_lr, 'Linear Regression', '#FF9800'),
    (axes[1], y_pred_nn, 'Neural Network',     '#1565C0'),
]:
    ax.scatter(y_test[sample]/1000, y_pred[sample]/1000, alpha=0.25, s=12, color=color)
    ax.plot([0,36],[0,36],'r--', lw=1.5, label='Perfect prediction')
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    ax.set_title(f'{name}\nR²={r2:.3f}  MAE=${mae:,.0f}', fontweight='bold')
    ax.set_xlabel('Actual ($k)'); ax.set_ylabel('Predicted ($k)')
    ax.set_xlim(0,36); ax.set_ylim(0,36); ax.legend()

plt.suptitle('Wide scatter — 77% of loan amount variance is unexplained by credit profile features', fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

## Prediction Accuracy Brackets

An intuitive way to communicate model performance to a non-technical audience: what fraction of predictions land within X% of the actual loan amount?

In [ ]:
errors_pct = np.abs(y_test - y_pred_nn) / y_test * 100
thresholds = [10, 20, 30, 50, 75]
within = [(errors_pct <= t).mean()*100 for t in thresholds]

for t, w in zip(thresholds, within):
    print(f'  Within ±{t:2d}%: {w:5.1f}% of predictions  ({int(w/100*len(y_test)):,} of {len(y_test):,} samples)')

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar([f'±{t}%' for t in thresholds], within, color='#1565C0', edgecolor='white', alpha=0.85)
for bar, w in zip(bars, within):
    ax.text(bar.get_x()+bar.get_width()/2, w+0.5, f'{w:.0f}%', ha='center', fontweight='bold')
ax.set_ylabel('% of predictions in bracket'); ax.set_ylim(0, 80)
ax.set_title('At ±30%: model lands within $3,000 of a $10,000 loan — usable as a first estimate', fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/accuracy_brackets.png', dpi=150, bbox_inches='tight')
plt.show()

## Residual Analysis

Where does the model consistently go wrong?

In [ ]:
residuals = y_test - y_pred_nn

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_pred_nn[sample]/1000, residuals[sample]/1000, alpha=0.25, s=12, color='#C62828')
axes[0].axhline(0, color='black', lw=1.5, linestyle='--')
axes[0].set_xlabel('Predicted ($k)'); axes[0].set_ylabel('Residual (Actual − Predicted, $k)')
axes[0].set_title('Model under-predicts large loans — it gravitates toward the mean', fontweight='bold')

axes[1].hist(residuals/1000, bins=60, color='#1565C0', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='red', lw=1.5, linestyle='--')
axes[1].axvline(residuals.mean()/1000, color='orange', lw=2, linestyle='--',
                label=f'Mean residual = ${residuals.mean():,.0f}')
axes[1].set_xlabel('Residual ($k)'); axes[1].set_ylabel('Count')
axes[1].set_title(f'Residuals roughly centred, std=${residuals.std():,.0f}', fontweight='bold')
axes[1].legend()
plt.tight_layout()
plt.savefig('../outputs/figures/residuals.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Mean residual: ${residuals.mean():,.0f}  (bias — are we systematically over/under?)')
print(f'Std residual:  ${residuals.std():,.0f}')
print(f'Max over-prediction:  ${(-residuals).max():,.0f}')
print(f'Max under-prediction: ${residuals.max():,.0f}')

### What the residuals tell me

The model systematically **under-predicts large loans**. When someone requests $30,000, the model might predict $18,000. This is regression-to-the-mean — the model hasn't seen enough evidence in the features to confidently predict a large amount, so it hedges toward the training average of $9,589.

This is expected given the low R². The model knows income, grade, and intent — but a high earner with grade A borrowing for home improvement could want anywhere from $5,000 to $35,000. Without more specific context (renovation cost, property value, etc.), the model can't distinguish.

## What Didn't Work

**Challenge 1 — Wrong task definition**
- **The Problem**: The Python code in the original repo trained a binary classifier predicting `loan_status` (0/1 default). But the Flutter app displayed results as 'Loan Amount: X' and the project is titled 'Loan Amount Prediction'. Classification and regression are fundamentally different problems.
- **What I Tried**: Initially kept the classification framing. But the Flutter output of '550034.9' (well outside [0,1]) confirmed the original intent was regression.
- **What Worked**: Rebuilding from scratch as regression — target = `loan_amnt`, linear output, MSE loss, regression metrics.

**Challenge 2 — Scaling the target was non-obvious**
- **The Problem**: Loan amounts range $500–$35,000. MSE on raw dollar values produces loss in the hundreds of millions. Training diverged immediately.
- **What I Tried**: Scaling only X. Loss stabilised but convergence was slow.
- **What Worked**: Scaling both X and y with StandardScaler. Loss dropped to normal range (~0.8–1.0) and the network trained cleanly. The key is remembering to inverse-transform predictions before showing them.
- **The Result**: Training converged in ~25 epochs instead of never.

**Challenge 3 — R² of 0.23 looks weak, but is it?**
- **The Problem**: R² of 0.23 means 77% of variance is unexplained. On first look this seems like model failure.
- **What I Investigated**: Correlation analysis shows no feature correlates above 0.34 with loan_amnt. The fundamental reason: loan amount is borrower-chosen based on personal need. Credit profile features measure risk capacity, not borrowing intent.
- **Conclusion**: This is a data limitation, not a modeling failure. More relevant features (intended use details, property value for home loans, medical procedure costs) would push R² much higher. The model captures the signal that exists.

## So What — In Plain English

Given a borrower's profile, the model predicts their loan amount within about ±$5,571 on average. For a typical $9,589 loan, that's a 58% margin — enough to give a rough bracket, not an exact figure.

Put another way: if someone with a $50,000 income applies for a home improvement loan with a B-grade profile, the model might predict $11,000. The actual loan could reasonably be anywhere from $5,000 to $25,000. The model narrows the range but doesn't nail it.

The Flutter app demonstrates this concept — enter a borrower profile, get a loan amount estimate. It should be treated as a starting-point estimate, not a definitive approval figure.

**Valid input ranges for realistic predictions:**
- `person_income`: $10,000 – $200,000 (annual, USD)
- `loan_amnt` (if used as reference): $500 – $35,000
- `loan_int_rate`: 5% – 24%
- `person_age`: 20 – 65
- Values outside these ranges will produce extrapolated results that are not reliable.

## What I'd Do Next

1. **Add loan-purpose-specific features** — the model can't distinguish a $5,000 medical emergency from a $5,000 vacation. If the dataset included more detail on *why* someone is borrowing, R² would improve substantially.

2. **Try gradient boosting (XGBoost / LightGBM)** — typically beats neural nets on tabular regression. The only reason I chose TF here was the TFLite mobile deployment requirement.

3. **Add prediction intervals** — instead of a point estimate, give a range (e.g., 'likely between $8,000 and $15,000'). Quantile regression or conformal prediction would make the model more honest and more useful.

4. **Re-evaluate with a temporal split** — this dataset has no timestamps, so I used random split. In production, you'd train on historical data and evaluate on more recent data to simulate real deployment conditions.

5. **Address the data/app inconsistency** — the original Flutter demo used inputs 14× outside the training distribution, producing nonsensical outputs. Proper input validation in the app (clamping to training range) would prevent this.